In [1]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.2 MB/s eta 0:00:00a 0:00:01


In [ ]:
!pip install -U --no-cache-dir numpy==1.26.4 scipy==1.11.4 scikit-learn==1.3.2


In [2]:
import os
import numpy as np
import shutil
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, accuracy_score

# Paths
dataset_path = "/kaggle/input/datasets/hjhjakd/skinfourth/skinfourth"
base_dir = "/kaggle/working/split_data"

# Create split directories
for split in ['train', 'val', 'test']:
    for class_name in os.listdir(dataset_path):
        os.makedirs(os.path.join(base_dir, split, class_name), exist_ok=True)

# Split data
for class_name in os.listdir(dataset_path):
    class_path = os.path.join(dataset_path, class_name)
    images = os.listdir(class_path)
    
    train_files, temp_files = train_test_split(images, test_size=0.3, random_state=42)
    val_files, test_files = train_test_split(temp_files, test_size=0.5, random_state=42)

    for file in train_files:
        shutil.copy(os.path.join(class_path, file), os.path.join(base_dir, 'train', class_name))
    for file in val_files:
        shutil.copy(os.path.join(class_path, file), os.path.join(base_dir, 'val', class_name))
    for file in test_files:
        shutil.copy(os.path.join(class_path, file), os.path.join(base_dir, 'test', class_name))

# Data generators
train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    os.path.join(base_dir, 'train'),
    target_size=(448, 448),
    batch_size=32,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    os.path.join(base_dir, 'val'),
    target_size=(448, 448),
    batch_size=32,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    os.path.join(base_dir, 'test'),
    target_size=(448, 448),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

Found 2677 images belonging to 21 classes.
Found 575 images belonging to 21 classes.
Found 582 images belonging to 21 classes.


In [3]:
# import os
import shutil

# Base dataset path
base_path = "/kaggle/working/split_data"

# Folders to remove

remove_classes = ["cb_main", "cd_main","ajb_main","jb_main"]

# Splits where classes exist
splits = ["train", "val", "test"]

for split in splits:
    for cls in remove_classes:
        dir_path = os.path.join(base_path, split, cls)
        if os.path.exists(dir_path):
            print(f"Removing: {dir_path}")
            shutil.rmtree(dir_path)
        else:
            print(f"Not found (skipped): {dir_path}")


Removing: /kaggle/working/split_data/train/cb_main
Removing: /kaggle/working/split_data/train/cd_main
Removing: /kaggle/working/split_data/train/ajb_main
Removing: /kaggle/working/split_data/train/jb_main
Removing: /kaggle/working/split_data/val/cb_main
Removing: /kaggle/working/split_data/val/cd_main
Removing: /kaggle/working/split_data/val/ajb_main
Removing: /kaggle/working/split_data/val/jb_main
Removing: /kaggle/working/split_data/test/cb_main
Removing: /kaggle/working/split_data/test/cd_main
Removing: /kaggle/working/split_data/test/ajb_main
Removing: /kaggle/working/split_data/test/jb_main


In [4]:
!pip install qiskit-machine-learning==0.8.3 --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 5.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 75.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 67.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 39.1 MB/s eta 0:00:00:00:0100:01


In [5]:
# =========================================================================
# STREAMLINED CORE PIPELINE: MOBILE...V3 + YOLOv11x -> LNN STACKING ENSEMBLE
# Logs Fold History (JSON), Confusion Matrices, and Exports ROC Arrays
# =========================================================================

import os
import time
import json
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.optim import AdamW
import torch.optim.lr_scheduler as lr_scheduler
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms
import torchvision.models as models

try:
    from ultralytics import YOLO
except ImportError:
    os.system('pip install ultralytics')
    from ultralytics import YOLO

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, roc_curve, confusion_matrix
from sklearn.preprocessing import label_binarize

# -------------------------------------------------------------------------
# 1. Environment Enforcements & Configuration Protocols
# -------------------------------------------------------------------------
if not torch.cuda.is_available():
    raise RuntimeError("🚨 GPU requested but CUDA accelerator is not detected by PyTorch!")

device = torch.device("cuda")
print(f"🚀 Dedicated Execution Platform: {device}\n")
torch.backends.cudnn.benchmark = True

DATA_ROOT = "/kaggle/working/split_data"

transform_standard = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# -------------------------------------------------------------------------
# 2. Dataset Map Structuring & Safe Global Extraction
# -------------------------------------------------------------------------
print("📁 Loading target image folder split architectures...")
train_dataset_raw = datasets.ImageFolder(os.path.join(DATA_ROOT, "train"), transform=transform_standard)
val_dataset_raw   = datasets.ImageFolder(os.path.join(DATA_ROOT, "val"), transform=transform_standard)
test_dataset      = datasets.ImageFolder(os.path.join(DATA_ROOT, "test"), transform=transform_standard)

cv_pool_dataset = torch.utils.data.ConcatDataset([train_dataset_raw, val_dataset_raw])
cv_targets = np.array([s[1] for s in train_dataset_raw.samples] + [s[1] for s in val_dataset_raw.samples])
num_classes = len(train_dataset_raw.classes)
test_labels = np.array([s[1] for s in test_dataset.samples])

print("⚡ Initializing Native YOLOv11x Backpropagation Tensors (1280 Dimensions)...")
yolo_native = YOLO("yolo11x-cls.pt")

def extract_yolo_embeddings(dataset_pool):
    embeddings = []
    samples = dataset_pool.datasets[0].samples + dataset_pool.datasets[1].samples if isinstance(dataset_pool, torch.utils.data.ConcatDataset) else dataset_pool.samples
    for img_path, _ in tqdm(samples, desc="YOLO Deep Embedding Extraction"):
        feat = yolo_native.embed(source=img_path, verbose=False)[0]
        embeddings.append(feat.view(-1).cpu())
    return torch.stack(embeddings).float()

cv_yolo_features = extract_yolo_embeddings(cv_pool_dataset)
test_yolo_features = extract_yolo_embeddings(test_dataset)

# --- Global Metrics Vault Structure for JSON serialization ---
metrics_vault = {
    "fold_histories": {
        "MobileNetV3-Large": {},
        "YOLOv11x": {},
        "LNN_Stacking_Ensemble": {}
    },
    "raw_roc_curves": {
        "LNN_Stacking_Ensemble": {}
    }
}

# -------------------------------------------------------------------------
# 3. Microarchitectures & Feature Isolation Maps
# -------------------------------------------------------------------------
class SimulatedQuantumRegularizationLayer(nn.Module):
    def __init__(self, num_qubits=8):
        super().__init__()
        self.num_qubits = num_qubits
        self.quantum_rx = nn.Parameter(torch.randn(num_qubits) * 0.02)
        self.quantum_ry = nn.Parameter(torch.randn(num_qubits) * 0.02)
    def forward(self, x):
        mapped_state = torch.sin(x * self.quantum_rx) + torch.cos(x * self.quantum_ry)
        entangled_state = torch.roll(mapped_state, shifts=1, dims=1) * mapped_state
        return torch.tanh(entangled_state)

class QuantumRegularizedYOLOAdapter(nn.Module):
    def __init__(self, input_dim=1280, num_classes=17, num_qubits=8):
        super().__init__()
        self.fc_reduce = nn.Linear(input_dim, 256)
        self.fc_to_quantum = nn.Linear(256, num_qubits)
        self.quantum_block = SimulatedQuantumRegularizationLayer(num_qubits=num_qubits)
        self.classifier_head = nn.Linear(256 + num_qubits, num_classes)
    def forward(self, feats, return_features=False):
        feats_reduced = nn.functional.relu(self.fc_reduce(feats))
        q_inputs = torch.tanh(self.fc_to_quantum(feats_reduced))
        q_features = self.quantum_block(q_inputs)
        hybrid_features = torch.cat([feats_reduced, q_features], dim=1)
        if return_features:
            return hybrid_features
        return self.classifier_head(hybrid_features)

def get_standalone_model(name, num_classes):
    if name == "MobileNetV3-Large":
        model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)
        return model.to(device)
    raise ValueError(f"Unsupported core framework model request: {name}")

# -------------------------------------------------------------------------
# 4. Engine Protocol Infrastructure (3-Fold Cross-Validation Setup)
# -------------------------------------------------------------------------
N_SPLITS = 3
BASE_EPOCHS = 6
YOLO_EPOCHS = 25  # Maintained convergence safety buffer
batch_size = 16
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

core_backbones = ["MobileNetV3-Large", "YOLOv11x"]
final_results_table = {}

cv_dense_store = {
    "MobileNetV3-Large": torch.zeros((len(cv_pool_dataset), 960)), 
    "YOLOv11x": torch.zeros((len(cv_pool_dataset), 264))
}
test_dense_store = {
    "MobileNetV3-Large": torch.zeros((len(test_dataset), 960)).to(device),
    "YOLOv11x": torch.zeros((len(test_dataset), 264)).to(device)
}

class LockedFeatureDataset(torch.utils.data.Dataset):
    def __init__(self, subset_indices, full_dataset, all_yolo_feats):
        self.subset_indices = subset_indices
        self.full_dataset = full_dataset
        self.all_yolo_feats = all_yolo_feats
    def __len__(self): return len(self.subset_indices)
    def __getitem__(self, idx):
        actual_pool_idx = self.subset_indices[idx]
        img, label = self.full_dataset[actual_pool_idx]
        return img, self.all_yolo_feats[actual_pool_idx], label

class TestLockedDataset(torch.utils.data.Dataset):
    def __init__(self, raw_test, test_feats):
        self.raw_test = raw_test
        self.test_feats = test_feats
    def __len__(self): return len(self.raw_test)
    def __getitem__(self, idx):
        img, lbl = self.raw_test[idx]
        return img, self.test_feats[idx], lbl

def compute_all_metrics(true_labels, predicted_probs):
    preds = np.argmax(predicted_probs, axis=1)
    acc = accuracy_score(true_labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(true_labels, preds, average='macro', zero_division=0)
    true_bin = label_binarize(true_labels, classes=list(range(num_classes)))
    try: auc_score = roc_auc_score(true_bin, predicted_probs, multi_class='ovr', average='macro')
    except: auc_score = 0.5
    return [acc, prec, rec, f1, f1, auc_score]

model_test_predictions = {}

# --- Train Core Stream Backbones ---
for model_name in core_backbones:
    print(f"\n🔄 Running 3-Fold Cross-Validation Framework Protocol for: {model_name}")
    oof_probs = np.zeros((len(cv_pool_dataset), num_classes))
    test_probs_accum = torch.zeros((len(test_dataset), num_classes)).to(device)
    total_inf_time = 0.0
    
    current_run_epochs = YOLO_EPOCHS if model_name == "YOLOv11x" else BASE_EPOCHS
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(cv_targets)), cv_targets)):
        train_loader = DataLoader(LockedFeatureDataset(train_idx, cv_pool_dataset, cv_yolo_features), batch_size=batch_size, shuffle=True, num_workers=2)
        val_loader   = DataLoader(LockedFeatureDataset(val_idx, cv_pool_dataset, cv_yolo_features), batch_size=batch_size, shuffle=False, num_workers=2)
        test_loader  = DataLoader(TestLockedDataset(test_dataset, test_yolo_features), batch_size=batch_size, shuffle=False, num_workers=2)
        
        if model_name == "YOLOv11x":
            model = QuantumRegularizedYOLOAdapter(input_dim=cv_yolo_features.shape[1], num_classes=num_classes).to(device)
            optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
        else:
            model = get_standalone_model(model_name, num_classes)
            optimizer = AdamW(model.parameters(), lr=4e-4, weight_decay=1e-4)
            
        criterion = nn.CrossEntropyLoss().to(device)
        
        fold_key = f"fold_{fold}"
        metrics_vault["fold_histories"][model_name][fold_key] = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
        
        for epoch in range(current_run_epochs):
            model.train()
            running_loss, correct_train, total_train = 0.0, 0, 0
            for imgs, feats, lbls in train_loader:
                optimizer.zero_grad()
                imgs, feats, lbls = imgs.to(device), feats.to(device), lbls.to(device)
                outputs = model(feats) if model_name == "YOLOv11x" else model(imgs)
                loss = criterion(outputs, lbls)
                loss.backward()
                optimizer.step()
                
                running_loss += loss.item() * imgs.size(0)
                _, predicted = torch.max(outputs, 1)
                total_train += lbls.size(0)
                correct_train += (predicted == lbls).sum().item()
            
            epoch_train_loss = running_loss / total_train
            epoch_train_acc = correct_train / total_train
            
            # Validation Step per Epoch for History Log tracking
            model.eval()
            val_loss_accum, correct_val, total_val = 0.0, 0, 0
            with torch.no_grad():
                for imgs, feats, lbls in val_loader:
                    imgs, feats, lbls = imgs.to(device), feats.to(device), lbls.to(device)
                    outputs = model(feats) if model_name == "YOLOv11x" else model(imgs)
                    loss = criterion(outputs, lbls)
                    val_loss_accum += loss.item() * imgs.size(0)
                    _, predicted = torch.max(outputs, 1)
                    total_val += lbls.size(0)
                    correct_val += (predicted == lbls).sum().item()
                    
            epoch_val_loss = val_loss_accum / total_val
            epoch_val_acc = correct_val / total_val
            
            metrics_vault["fold_histories"][model_name][fold_key]["train_loss"].append(epoch_train_loss)
            metrics_vault["fold_histories"][model_name][fold_key]["train_acc"].append(epoch_train_acc)
            metrics_vault["fold_histories"][model_name][fold_key]["val_loss"].append(epoch_val_loss)
            metrics_vault["fold_histories"][model_name][fold_key]["val_acc"].append(epoch_val_acc)

        # Capture Dense Features on Final Validation State Pass
        model.eval()
        idx_counter = 0
        with torch.no_grad():
            for imgs, feats, lbls in val_loader:
                imgs, feats = imgs.to(device), feats.to(device)
                bs = imgs.size(0)
                if model_name == "YOLOv11x":
                    out = model(feats)
                    cv_dense_store["YOLOv11x"][val_idx[idx_counter:idx_counter+bs]] = model(feats, return_features=True).cpu()
                else:
                    latents = model.features(imgs)
                    cv_dense_store["MobileNetV3-Large"][val_idx[idx_counter:idx_counter+bs]] = model.avgpool(latents).flatten(1).cpu()
                    out = model(imgs)
                    
                softmax_res = nn.functional.softmax(out, dim=1).cpu().numpy()
                oof_probs[val_idx[idx_counter : idx_counter + bs]] = softmax_res
                idx_counter += bs
                
            start_inf = time.time()
            test_idx_count = 0
            for imgs, feats, _ in test_loader:
                imgs, feats = imgs.to(device), feats.to(device)
                curr_bs = imgs.size(0)
                if model_name == "YOLOv11x":
                    out_test = model(feats)
                    test_dense_store["YOLOv11x"][test_idx_count:test_idx_count+curr_bs] += model(feats, return_features=True)
                else:
                    latents = model.features(imgs)
                    test_dense_store["MobileNetV3-Large"][test_idx_count:test_idx_count+curr_bs] += model.avgpool(latents).flatten(1)
                    out_test = model(imgs)
                    
                test_probs_accum[test_idx_count : test_idx_count + curr_bs] += nn.functional.softmax(out_test, dim=1)
                test_idx_count += curr_bs
            total_inf_time += (time.time() - start_inf)

    avg_test_probs = (test_probs_accum / N_SPLITS).cpu().numpy()
    final_results_table[model_name] = compute_all_metrics(test_labels, avg_test_probs) + [total_inf_time / N_SPLITS]
    model_test_predictions[model_name] = np.argmax(avg_test_probs, axis=1)
    
    test_dense_store[model_name] /= N_SPLITS

# -------------------------------------------------------------------------
# 5. Formulate Concatenated Latent Vector Planes (1224 Dimensions)
# -------------------------------------------------------------------------
print("\n🧬 Formulating Concatenated Latent Vector Planes (Input Dimension = 1224)...")
meta_X_train = torch.cat([cv_dense_store["MobileNetV3-Large"], cv_dense_store["YOLOv11x"]], dim=1)
meta_y_train = torch.tensor(cv_targets, dtype=torch.long)
meta_X_test = torch.cat([test_dense_store["MobileNetV3-Large"], test_dense_store["YOLOv11x"]], dim=1)

X_train_meta_np = meta_X_train.numpy()
y_train_meta_np = meta_y_train.numpy()

# -------------------------------------------------------------------------
# 6. PROPOSED METHOD: High-Capacity Continuous Liquid Neural Head
# -------------------------------------------------------------------------
class OptimizedLiquidCell(nn.Module):
    def __init__(self, in_features, out_features, dt=0.03):
        super().__init__()
        self.dt = dt
        self.w_state = nn.Linear(in_features, out_features)
        self.w_leak = nn.Parameter(torch.abs(torch.randn(out_features)) * 0.04)
        self.layernorm = nn.LayerNorm(out_features)
    def forward(self, x, h_prev):
        derivative = torch.tanh(self.w_state(x)) - (h_prev * self.w_leak)
        return self.layernorm(h_prev + (self.dt * derivative))

class DeepLNNMetaClassifier(nn.Module):
    def __init__(self, input_dim, num_classes, steps=10):
        super().__init__()
        self.steps = steps
        self.input_layer = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(p=0.35)
        )
        self.lnn_cell = OptimizedLiquidCell(256, 256)
        self.output_head = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(p=0.25),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        x_proj = self.input_layer(x)
        h = torch.zeros(x.size(0), 256).to(x.device)
        for _ in range(self.steps):
            h = self.lnn_cell(x_proj, h)
        return self.output_head(h)

print("🧠 Optimizing High-Capacity Deep LNN (Proposed Model) over 3-Fold Metadata splits...")
meta_skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
lnn_test_probs_accum = np.zeros((len(test_dataset), num_classes))
lnn_inf_time_total = 0.0

for fold, (m_train_idx, m_val_idx) in enumerate(meta_skf.split(X_train_meta_np, y_train_meta_np)):
    fold_key = f"fold_{fold}"
    metrics_vault["fold_histories"]["LNN_Stacking_Ensemble"][fold_key] = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    
    m_train_dataset = TensorDataset(meta_X_train[m_train_idx].to(device), meta_y_train[m_train_idx].to(device))
    m_val_dataset   = TensorDataset(meta_X_train[m_val_idx].to(device), meta_y_train[m_val_idx].to(device))
    
    m_train_loader = DataLoader(m_train_dataset, batch_size=32, shuffle=True)
    m_val_loader   = DataLoader(m_val_dataset, batch_size=32, shuffle=False)
    
    lnn_meta = DeepLNNMetaClassifier(input_dim=1224, num_classes=num_classes).to(device)
    optimizer_lnn = AdamW(lnn_meta.parameters(), lr=2e-3, weight_decay=1e-2)
    criterion_lnn = nn.CrossEntropyLoss().to(device)
    
    for epoch in range(45):
        lnn_meta.train()
        r_loss, c_train, t_train = 0.0, 0, 0
        for xb, yb in m_train_loader:
            optimizer_lnn.zero_grad()
            outputs = lnn_meta(xb)
            loss = criterion_lnn(outputs, yb)
            loss.backward()
            optimizer_lnn.step()
            
            r_loss += loss.item() * xb.size(0)
            _, preds = torch.max(outputs, 1)
            t_train += yb.size(0)
            c_train += (preds == yb).sum().item()
            
        # Meta-Validation Step 
        lnn_meta.eval()
        v_loss, c_val, t_val = 0.0, 0, 0
        with torch.no_grad():
            for xb, yb in m_val_loader:
                outputs = lnn_meta(xb)
                loss = criterion_lnn(outputs, yb)
                v_loss += loss.item() * xb.size(0)
                _, preds = torch.max(outputs, 1)
                t_val += yb.size(0)
                c_val += (preds == yb).sum().item()
                
        metrics_vault["fold_histories"]["LNN_Stacking_Ensemble"][fold_key]["train_loss"].append(r_loss / t_train)
        metrics_vault["fold_histories"]["LNN_Stacking_Ensemble"][fold_key]["train_acc"].append(c_train / t_train)
        metrics_vault["fold_histories"]["LNN_Stacking_Ensemble"][fold_key]["val_loss"].append(v_loss / t_val)
        # --- FIXED PATH BELOW ---
        metrics_vault["fold_histories"]["LNN_Stacking_Ensemble"][fold_key]["val_acc"].append(c_val / t_val)

    lnn_meta.eval()
    start_lnn = time.time()
    with torch.no_grad():
        probs = nn.functional.softmax(lnn_meta(meta_X_test.to(device)), dim=1).cpu().numpy()
    lnn_inf_time_total += (time.time() - start_lnn)
    lnn_test_probs_accum += probs

final_lnn_probs = lnn_test_probs_accum / N_SPLITS
final_results_table["LNN Stacking Ensemble (Proposed)"] = compute_all_metrics(test_labels, final_lnn_probs) + [lnn_inf_time_total / N_SPLITS]
model_test_predictions["LNN Stacking Ensemble (Proposed)"] = np.argmax(final_lnn_probs, axis=1)

# --- Compute and Save Raw ROC Curves Threshold Arrays for LNN ---
test_labels_bin = label_binarize(test_labels, classes=list(range(num_classes)))
for c in range(num_classes):
    fpr, tpr, thresholds = roc_curve(test_labels_bin[:, c], final_lnn_probs[:, c])
    metrics_vault["raw_roc_curves"]["LNN_Stacking_Ensemble"][f"class_{c}"] = {
        "fpr": fpr.tolist(),
        "tpr": tpr.tolist(),
        "thresholds": thresholds.tolist()
    }

# -------------------------------------------------------------------------
# 7. Generation of Analytics & Storage Archives
# -------------------------------------------------------------------------
print("\n💾 Serializing analytical telemetry files to JSON format...")
with open("experiment_metrics_vault.json", "w") as j_file:
    json.dump(metrics_vault, j_file, indent=4)
print("✨ Complete execution training tracking saved successfully to 'experiment_metrics_vault.json'")

print("\n🧩 Calculating Class Confusion Matrices...")
for target_m in ["MobileNetV3-Large", "YOLOv11x", "LNN Stacking Ensemble (Proposed)"]:
    raw_cm = confusion_matrix(test_labels, model_test_predictions[target_m])
    normalized_cm = raw_cm.astype('float') / raw_cm.sum(axis=1)[:, np.newaxis]
    print(f"\n📊 Normalized Confusion Matrix for Model: {target_m}")
    print(np.array2string(normalized_cm, formatter={'float_kind': lambda x: f"{x:.2f}"}))

# -------------------------------------------------------------------------
# 8. Print Final Consolidated Matrix Report
# -------------------------------------------------------------------------
print("\n" + "="*116)
print(f"{'Algorithm Model Architecture':<32} | {'Acc':<6} | {'Prec':<6} | {'Recall':<6} | {'F1':<6} | {'Macro-F1':<8} | {'AUC-ROC':<7} | {'Latency(s)':<8}")
print("="*116)
for key, vals in final_results_table.items():
    print(f"{key:<32} | {vals[0]:.4f} | {vals[1]:.4f} | {vals[2]:.4f} | {vals[3]:.4f} | {vals[4]:.4f}   | {vals[5]:.4f}  | {vals[6]:.5f}")
print("="*116)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🚀 Dedicated Execution Platform: cuda

📁 Loading target image folder split architectures...
⚡ Initializing Native YOLOv11x Backpropagation Tensors (1280 Dimensions)...


YOLO Deep Embedding Extraction: 100%|██████████| 460/460 [00:18<00:00, 25.38it/s]



🔄 Running 3-Fold Cross-Validation Framework Protocol for: MobileNetV3-Large
Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 131MB/s] 



🔄 Running 3-Fold Cross-Validation Framework Protocol for: YOLOv11x

🧬 Formulating Concatenated Latent Vector Planes (Input Dimension = 1224)...
🧠 Optimizing High-Capacity Deep LNN (Proposed Model) over 3-Fold Metadata splits...

💾 Serializing analytical telemetry files to JSON format...
✨ Complete execution training tracking saved successfully to 'experiment_metrics_vault.json'

🧩 Calculating Class Confusion Matrices...

📊 Normalized Confusion Matrix for Model: MobileNetV3-Large
[[0.92 0.00 0.00 0.00 0.00 0.00 0.00 0.00 0.08 0.00 0.00 0.00 0.00 0.00 0.00 0.00 0.00]
 [0.04 0.82 0.00 0.00 0.04 0.00 0.04 0.00 0.00 0.00 0.00 0.00 0.00 0.04 0.00 0.04 0.00]
 [0.00 0.04 0.87 0.00 0.04 0.00 0.04 0.00 0.00 0.00 0.00 0.00 0.00 0.00 0.00 0.00 0.00]
 [0.00 0.00 0.00 0.90 0.03 0.03 0.00 0.00 0.00 0.00 0.00 0.03 0.00 0.00 0.00 0.00 0.00]
 [0.00 0.00 0.00 0.00 0.90 0.00 0.03 0.00 0.00 0.00 0.00 0.00 0.00 0.00 0.07 0.00 0.00]
 [0.00 0.00 0.00 0.00 0.00 0.97 0.00 0.00 0.00 0.00 0.00 0.00 0.00 0.00 0.0